# 同任务横评 · 4 种框架实现「研究 Agent」

**任务**：给定主题，agent 自主决定调用以下工具，最终生成 200 字以内简介：

- `search(query)`：返回若干段落（mock 知识库，沿用第 04 章 DOCS）。
- `now()`：当前北京时间。

我们用 4 种风格实现并对比：

1. 原生 Anthropic SDK
2. LangGraph
3. LlamaIndex Workflows
4. AutoGen 0.4

（注：AutoGen / LlamaIndex 版本变动较快，运行时如 API 不一致请按官方最新文档微调。）

In [ ]:
import os, sys, json, datetime
from zoneinfo import ZoneInfo
import numpy as np
sys.path.append(os.path.abspath('../..'))

DOCS = [
    'ReAct（Yao 2022）让 LLM 交替输出 Thought/Action/Observation。',
    'Reflexion（Shinn 2023）通过自然语言反思在多 episode 间积累经验。',
    'LATS（Zhou 2024）= ToT + ReAct + Reflexion，用 MCTS 串起搜索/行动/反思。',
    'MCP（Anthropic 2024）把 LLM 与工具/数据接口标准化。',
    'GRPO（DeepSeek 2024）省掉 value model，用组内归一 advantage。',
    'SkyRL-Agent（2025）多轮长程 agent RL 训练框架。',
    'Claude Computer Use（2024）让模型直接操作屏幕、键鼠。',
    'MapAgent（2025）分层多 agent 框架，map-tool agent 并行调地图 API。',
]

def search(query: str, k: int = 3) -> list[str]:
    # 简易关键词匹配，避免本 notebook 必须装 sentence-transformers
    scored = sorted(DOCS, key=lambda d: -sum(1 for w in query if w in d))
    return scored[:k]

def now() -> str:
    return datetime.datetime.now(ZoneInfo('Asia/Shanghai')).isoformat(timespec='seconds')

## 1. 原生 Anthropic SDK

In [ ]:
from anthropic import Anthropic
anthropic = Anthropic()
MODEL = 'claude-sonnet-4-5'

TOOLS = [
    {'name': 'search', 'description': '检索知识库片段',
     'input_schema': {'type': 'object', 'properties': {'query': {'type': 'string'}, 'k': {'type': 'integer'}}, 'required': ['query']}},
    {'name': 'now', 'description': '当前北京时间',
     'input_schema': {'type': 'object', 'properties': {}}},
]
DISPATCH = {'search': lambda **kw: search(**kw), 'now': lambda: now()}

def native_research(topic: str, max_steps: int = 5):
    sys_p = '你是研究助理。可调用 search/now，最终输出 200 字内中文简介。'
    msgs = [{'role': 'user', 'content': f'主题：{topic}'}]
    in_t = out_t = 0
    for _ in range(max_steps):
        r = anthropic.messages.create(model=MODEL, max_tokens=512, system=sys_p, tools=TOOLS, messages=msgs)
        in_t += r.usage.input_tokens; out_t += r.usage.output_tokens
        if r.stop_reason != 'tool_use':
            return ''.join(b.text for b in r.content if b.type == 'text'), in_t, out_t
        msgs.append({'role': 'assistant', 'content': r.content})
        results = []
        for b in r.content:
            if b.type == 'tool_use':
                out = DISPATCH[b.name](**b.input)
                results.append({'type': 'tool_result', 'tool_use_id': b.id, 'content': json.dumps(out, ensure_ascii=False)})
        msgs.append({'role': 'user', 'content': results})
    return '[max]', in_t, out_t

ans, ti, to = native_research('2025 LLM Agent RL 训练进展')
print('Native:', ans, '\n', ti, to)

## 2. LangGraph 实现

In [ ]:
from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, END
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool

@tool
def lc_search(query: str, k: int = 3) -> list[str]:
    """检索知识库片段"""
    return search(query, k)

@tool
def lc_now() -> str:
    """当前北京时间"""
    return now()

from langgraph.prebuilt import create_react_agent

lg_agent = create_react_agent(
    ChatAnthropic(model=MODEL, temperature=0),
    tools=[lc_search, lc_now],
    state_modifier='你是研究助理，输出 200 字内中文简介。',
)

out = lg_agent.invoke({'messages': [('user', '2025 LLM Agent RL 训练进展')]})
print('LangGraph:', out['messages'][-1].content)

## 3. LlamaIndex Workflows

In [ ]:
# LlamaIndex 0.11+ 提供 ReActAgent，是迁移 ReAct 最简洁的方式
from llama_index.llms.anthropic import Anthropic as LIAnthropic
from llama_index.core.tools import FunctionTool
from llama_index.core.agent import ReActAgent

tools = [
    FunctionTool.from_defaults(fn=search, name='search', description='检索知识库片段'),
    FunctionTool.from_defaults(fn=now, name='now', description='当前北京时间'),
]
li_agent = ReActAgent.from_tools(tools, llm=LIAnthropic(model=MODEL, temperature=0), verbose=False)
resp = li_agent.chat('请介绍 2025 LLM Agent RL 训练进展，200 字内。')
print('LlamaIndex:', str(resp))

## 4. AutoGen 0.4

In [ ]:
# AutoGen 0.4+ API（异步）。如果你装的是 0.2.x 旧版，请用 ConversableAgent + register_function。
import asyncio
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console
from autogen_agentchat.messages import TextMessage
from autogen_ext.models.anthropic import AnthropicChatCompletionClient

async def run_autogen():
    model_client = AnthropicChatCompletionClient(model=MODEL)
    agent = AssistantAgent(
        name='researcher',
        model_client=model_client,
        tools=[search, now],
        system_message='你是研究助理，输出 200 字内中文简介。',
    )
    result = await agent.on_messages(
        [TextMessage(content='2025 LLM Agent RL 训练进展', source='user')],
        cancellation_token=None,
    )
    print('AutoGen:', result.chat_message.content)

# Jupyter 可直接 await
await run_autogen()

## 5. 主观横评（填表）

| 维度 | 原生 SDK | LangGraph | LlamaIndex | AutoGen 0.4 |
|------|---------|-----------|------------|-------------|
| 行数 | 中 | 短（prebuilt） | 短 | 中 |
| 可控性 | ★★★★★ | ★★★★☆ | ★★★☆☆ | ★★★☆☆ |
| 调试 trace | ★★☆☆☆（自己 print） | ★★★★☆（LangSmith） | ★★★☆☆ | ★★★☆☆ |
| 适合复杂状态 | ★★☆☆☆ | ★★★★★ | ★★★☆☆ | ★★★★☆ |
| 学习曲线 | 平 | 中 | 中-陡 | 中-陡 |

## 思考

- 单 agent + 简单 tool：直接用原生 SDK，最少依赖。
- 复杂状态/多 agent/HITL：LangGraph 显式图最稳。
- RAG 主线：LlamaIndex 抽象更天然。
- 对话式多角色协作 + 想要快速试 idea：AutoGen 上手快。

## 进阶

1. 给每个版本加 LangSmith / Langfuse trace，对比可观测性。
2. 在同一个任务集（10 题）上跑 4 种实现，统计 token / 时延 / 准确率。
3. 用 DSPy 把 prompt 自动优化一遍，对比手写 vs 优化后效果。